<a href="https://colab.research.google.com/github/Ebrahim488/Ebrahim488/blob/main/First_Proj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# AI SECURITY SURVEILLANCE SYSTEM
# 3 ROBoflow MODELS + MULTI OBJECT TRACKING
#
# Models:
#   1. Military Uniform
#   2. Weapon
#   3. Vehicle
#
# Detection: Every 2 frames
# Tracking: Every frame
# ============================================================


# ============================================================
# 1) INSTALL
# ============================================================

!pip install -q requests opencv-python


# ============================================================
# 2) IMPORTS
# ============================================================

import cv2
import requests
import time
import os
import base64
import numpy as np

from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

from google.colab import files
from IPython.display import display, HTML


# ============================================================
# 3) ROBOFLOW CONFIGURATION
# ============================================================

ROBOFLOW_API_KEY = "fIV9Dd1SB5yvGewfOhQB"

# ------------------------------------------------------------
# SAME 3 MODELS
# ------------------------------------------------------------

UNIFORM_MODEL = "military-soldier/1"

WEAPON_MODEL = (
    "security-and-weapon-detection-d4ya2-yyieq-nuyqk-v1n7e/1"
)

VEHICLE_MODEL = "toofan-llvuy/1"


# ============================================================
# 4) DETECTION SETTINGS
# ============================================================

CONF_THRESHOLD = 0.25

# AI detection every 2 frames
DETECTION_INTERVAL = 2


# ============================================================
# 5) TRACKING SETTINGS
# ============================================================

# Number of frames a tracker can survive without detection
MAX_LOST_FRAMES = 12

# IoU matching threshold
IOU_THRESHOLD = 0.20

# Maximum center-distance ratio
MAX_CENTER_DISTANCE = 0.35


# ============================================================
# 6) PERFORMANCE SETTINGS
# ============================================================

MAX_API_WIDTH = 960

JPEG_QUALITY = 80

API_TIMEOUT = 45

# Three models run simultaneously
MAX_WORKERS = 3


# ============================================================
# 7) OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = Path(
    "integrated_tracking"
)

OUTPUT_DIR.mkdir(
    exist_ok=True
)

INPUT_VIDEO = (
    OUTPUT_DIR /
    "input.mp4"
)

RAW_OUTPUT_VIDEO = (
    OUTPUT_DIR /
    "tracking_raw.mp4"
)

FINAL_OUTPUT_VIDEO = (
    OUTPUT_DIR /
    "tracking_final.mp4"
)


# ============================================================
# 8) DISPLAY CONFIG
# ============================================================

print("=" * 70)
print("AI SECURITY SURVEILLANCE SYSTEM")
print("=" * 70)

print("\nModels:")
print("Uniform :", UNIFORM_MODEL)
print("Weapon  :", WEAPON_MODEL)
print("Vehicle :", VEHICLE_MODEL)

print("\nSettings:")
print("Confidence       :", CONF_THRESHOLD)
print("Detection every  :", DETECTION_INTERVAL, "frames")
print("Max lost frames  :", MAX_LOST_FRAMES)
print("IoU threshold    :", IOU_THRESHOLD)
print("API width        :", MAX_API_WIDTH)


# ============================================================
# 9) UPLOAD VIDEO
# ============================================================

print("\n" + "=" * 70)
print("UPLOAD YOUR VIDEO")
print("=" * 70)

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No video uploaded.")


uploaded_name = next(iter(uploaded))

with open(
    INPUT_VIDEO,
    "wb"
) as f:

    f.write(
        uploaded[uploaded_name]
    )


print(
    "\nVideo uploaded:",
    uploaded_name
)


# ============================================================
# 10) OPEN VIDEO
# ============================================================

cap = cv2.VideoCapture(
    str(INPUT_VIDEO)
)

if not cap.isOpened():

    raise RuntimeError(
        "Cannot open video."
    )


fps = cap.get(
    cv2.CAP_PROP_FPS
)

if fps <= 0:
    fps = 25.0


frame_width = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

frame_height = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)

total_frames = int(
    cap.get(
        cv2.CAP_PROP_FRAME_COUNT
    )
)


duration = (
    total_frames / fps
    if fps > 0
    else 0
)


print("\nVideo information:")
print(
    f"Resolution : {frame_width} x {frame_height}"
)

print(
    f"FPS        : {fps:.2f}"
)

print(
    f"Frames     : {total_frames}"
)

print(
    f"Duration   : {duration:.2f} sec"
)


# ============================================================
# 11) VIDEO WRITER
# ============================================================

fourcc = cv2.VideoWriter_fourcc(
    *"mp4v"
)

writer = cv2.VideoWriter(

    str(RAW_OUTPUT_VIDEO),

    fourcc,

    fps,

    (
        frame_width,
        frame_height
    )
)


if not writer.isOpened():

    cap.release()

    raise RuntimeError(
        "Cannot create output video."
    )


# ============================================================
# 12) RESIZE FRAME FOR API
# ============================================================

def resize_for_api(frame):

    h, w = frame.shape[:2]

    if w <= MAX_API_WIDTH:

        return frame

    scale = (
        MAX_API_WIDTH / w
    )

    new_w = MAX_API_WIDTH

    new_h = int(
        h * scale
    )

    return cv2.resize(

        frame,

        (
            new_w,
            new_h
        ),

        interpolation=cv2.INTER_AREA
    )


# ============================================================
# 13) ENCODE FRAME
# ============================================================

def encode_frame(frame):

    success, buffer = cv2.imencode(

        ".jpg",

        frame,

        [
            int(
                cv2.IMWRITE_JPEG_QUALITY
            ),
            JPEG_QUALITY
        ]
    )

    if not success:

        raise RuntimeError(
            "Frame encoding failed."
        )

    return buffer.tobytes()


# ============================================================
# 14) RUN ROBOFLOW MODEL
# ============================================================

def run_roboflow(
    image_bytes,
    model_id
):

    endpoint = (
        "https://detect.roboflow.com/"
        f"{model_id}"
    )

    params = {

        "api_key":
            ROBOFLOW_API_KEY,

        "confidence":
            CONF_THRESHOLD
    }


    try:

        response = requests.post(

            endpoint,

            params=params,

            files={

                "file": (

                    "frame.jpg",

                    image_bytes,

                    "image/jpeg"

                )
            },

            timeout=API_TIMEOUT
        )


        if response.status_code != 200:

            return {

                "predictions": [],

                "error":
                    response.text[:500]

            }


        return response.json()


    except Exception as e:

        return {

            "predictions": [],

            "error":
                str(e)

        }


# ============================================================
# 15) IOU
# ============================================================

def calculate_iou(
    box_a,
    box_b
):

    ax1, ay1, ax2, ay2 = box_a

    bx1, by1, bx2, by2 = box_b


    ix1 = max(
        ax1,
        bx1
    )

    iy1 = max(
        ay1,
        by1
    )

    ix2 = min(
        ax2,
        bx2
    )

    iy2 = min(
        ay2,
        by2
    )


    iw = max(
        0,
        ix2 - ix1
    )

    ih = max(
        0,
        iy2 - iy1
    )


    intersection = (
        iw * ih
    )


    area_a = (

        max(
            0,
            ax2 - ax1
        )
        *
        max(
            0,
            ay2 - ay1
        )
    )


    area_b = (

        max(
            0,
            bx2 - bx1
        )
        *
        max(
            0,
            by2 - by1
        )
    )


    union = (
        area_a
        + area_b
        - intersection
    )


    if union <= 0:
        return 0.0


    return (
        intersection / union
    )


# ============================================================
# 16) CENTER DISTANCE
# ============================================================

def center_distance_ratio(
    box_a,
    box_b
):

    ax1, ay1, ax2, ay2 = box_a

    bx1, by1, bx2, by2 = box_b


    acx = (
        ax1 + ax2
    ) / 2

    acy = (
        ay1 + ay2
    ) / 2


    bcx = (
        bx1 + bx2
    ) / 2

    bcy = (
        by1 + by2
    ) / 2


    distance = np.sqrt(

        (
            acx - bcx
        ) ** 2

        +

        (
            acy - bcy
        ) ** 2
    )


    aw = max(
        1,
        ax2 - ax1
    )

    ah = max(
        1,
        ay2 - ay1
    )


    diagonal = np.sqrt(

        aw ** 2
        +
        ah ** 2
    )


    return (
        distance / diagonal
    )


# ============================================================
# 17) TRACK CLASS
# ============================================================

class Track:

    def __init__(
        self,
        bbox,
        confidence,
        class_name,
        track_id
    ):

        self.track_id = track_id

        self.bbox = np.array(
            bbox,
            dtype=np.float32
        )

        self.confidence = confidence

        self.class_name = class_name

        self.lost = 0

        self.age = 1

        self.hits = 1


        # ----------------------------------------------------
        # Kalman Filter
        #
        # State:
        # cx, cy, width, height,
        # vx, vy, vw, vh
        # ----------------------------------------------------

        self.kalman = cv2.KalmanFilter(
            8,
            4
        )


        self.kalman.transitionMatrix = np.array(

            [

                [1,0,0,0,1,0,0,0],

                [0,1,0,0,0,1,0,0],

                [0,0,1,0,0,0,1,0],

                [0,0,0,1,0,0,0,1],

                [0,0,0,0,1,0,0,0],

                [0,0,0,0,0,1,0,0],

                [0,0,0,0,0,0,1,0],

                [0,0,0,0,0,0,0,1]

            ],

            dtype=np.float32
        )


        self.kalman.measurementMatrix = np.array(

            [

                [1,0,0,0,0,0,0,0],

                [0,1,0,0,0,0,0,0],

                [0,0,1,0,0,0,0,0],

                [0,0,0,1,0,0,0,0]

            ],

            dtype=np.float32
        )


        self.kalman.processNoiseCov = (

            np.eye(
                8,
                dtype=np.float32
            )
            * 0.03
        )


        self.kalman.measurementNoiseCov = (

            np.eye(
                4,
                dtype=np.float32
            )
            * 0.08
        )


        self.kalman.errorCovPost = (

            np.eye(
                8,
                dtype=np.float32
            )
        )


        x1, y1, x2, y2 = self.bbox


        cx = (
            x1 + x2
        ) / 2

        cy = (
            y1 + y2
        ) / 2

        w = (
            x2 - x1
        )

        h = (
            y2 - y1
        )


        self.kalman.statePost = np.array(

            [
                [cx],
                [cy],
                [w],
                [h],
                [0],
                [0],
                [0],
                [0]
            ],

            dtype=np.float32
        )


    # ========================================================
    # PREDICT
    # ========================================================

    def predict(self):

        prediction = (
            self.kalman.predict()
        )


        cx = float(
            prediction[0]
        )

        cy = float(
            prediction[1]
        )

        w = max(
            2.0,
            float(
                prediction[2]
            )
        )

        h = max(
            2.0,
            float(
                prediction[3]
            )
        )


        self.bbox = np.array(

            [

                cx - w / 2,

                cy - h / 2,

                cx + w / 2,

                cy + h / 2

            ],

            dtype=np.float32
        )


        self.age += 1

        self.lost += 1


        return self.bbox


    # ========================================================
    # UPDATE
    # ========================================================

    def update(
        self,
        bbox,
        confidence,
        class_name
    ):

        x1, y1, x2, y2 = bbox


        cx = (
            x1 + x2
        ) / 2

        cy = (
            y1 + y2
        ) / 2

        w = (
            x2 - x1
        )

        h = (
            y2 - y1
        )


        measurement = np.array(

            [
                [cx],
                [cy],
                [w],
                [h]
            ],

            dtype=np.float32
        )


        self.kalman.correct(
            measurement
        )


        self.bbox = np.array(

            bbox,

            dtype=np.float32
        )


        self.confidence = confidence

        self.class_name = class_name

        self.lost = 0

        self.hits += 1


# ============================================================
# 18) MULTI OBJECT TRACKER
# ============================================================

class MultiObjectTracker:

    def __init__(
        self,
        name
    ):

        self.name = name

        self.tracks = []

        self.next_id = 1


    # ========================================================
    # PREDICT
    # ========================================================

    def predict(self):

        for track in self.tracks:

            track.predict()


    # ========================================================
    # UPDATE
    # ========================================================

    def update(
        self,
        detections
    ):

        # ----------------------------------------------------
        # No existing tracks
        # ----------------------------------------------------

        if len(self.tracks) == 0:

            for detection in detections:

                self.create_track(
                    detection
                )

            return


        # ----------------------------------------------------
        # Candidate matches
        # ----------------------------------------------------

        candidates = []


        for ti, track in enumerate(
            self.tracks
        ):

            for di, detection in enumerate(
                detections
            ):

                # Same class gets preference
                if (

                    track.class_name
                    != detection["class"]

                ):

                    continue


                iou = calculate_iou(

                    track.bbox,

                    detection["bbox"]
                )


                distance = (
                    center_distance_ratio(

                        track.bbox,

                        detection["bbox"]
                    )
                )


                if (

                    iou >= IOU_THRESHOLD

                    or

                    distance
                    <= MAX_CENTER_DISTANCE

                ):

                    score = (

                        iou

                        -
                        0.15 * distance
                    )


                    candidates.append(

                        (
                            score,
                            ti,
                            di
                        )
                    )


        # Highest score first
        candidates.sort(
            reverse=True
        )


        used_tracks = set()

        used_detections = set()


        # ----------------------------------------------------
        # MATCH
        # ----------------------------------------------------

        for (

            score,
            ti,
            di

        ) in candidates:


            if ti in used_tracks:
                continue


            if di in used_detections:
                continue


            self.tracks[
                ti
            ].update(

                detections[
                    di
                ]["bbox"],

                detections[
                    di
                ]["confidence"],

                detections[
                    di
                ]["class"]

            )


            used_tracks.add(
                ti
            )

            used_detections.add(
                di
            )


        # ----------------------------------------------------
        # REMOVE OLD TRACKS
        # ----------------------------------------------------

        self.tracks = [

            track

            for track in self.tracks

            if track.lost
            <= MAX_LOST_FRAMES

        ]


        # ----------------------------------------------------
        # CREATE NEW TRACKS
        # ----------------------------------------------------

        for di, detection in enumerate(
            detections
        ):

            if di not in used_detections:

                self.create_track(
                    detection
                )


    # ========================================================
    # CREATE TRACK
    # ========================================================

    def create_track(
        self,
        detection
    ):

        track = Track(

            detection["bbox"],

            detection["confidence"],

            detection["class"],

            self.next_id

        )


        self.next_id += 1


        self.tracks.append(
            track
        )


# ============================================================
# 19) CREATE 3 TRACKERS
# ============================================================

uniform_tracker = (
    MultiObjectTracker(
        "UNIFORM"
    )
)


weapon_tracker = (
    MultiObjectTracker(
        "WEAPON"
    )
)


vehicle_tracker = (
    MultiObjectTracker(
        "VEHICLE"
    )
)


# ============================================================
# 20) CONVERT ROBOFLOW BOXES
# ============================================================

def convert_predictions(

    predictions,

    original_width,
    original_height,

    api_width,
    api_height

):

    results = []


    scale_x = (

        original_width
        /
        api_width

    )


    scale_y = (

        original_height
        /
        api_height

    )


    for prediction in predictions:


        confidence = float(

            prediction.get(
                "confidence",
                0
            )

        )


        if confidence < CONF_THRESHOLD:
            continue


        class_name = prediction.get(

            "class",

            "unknown"

        )


        x = float(
            prediction["x"]
        )

        y = float(
            prediction["y"]
        )

        w = float(
            prediction["width"]
        )

        h = float(
            prediction["height"]
        )


        x1 = (

            x - w / 2

        ) * scale_x


        y1 = (

            y - h / 2

        ) * scale_y


        x2 = (

            x + w / 2

        ) * scale_x


        y2 = (

            y + h / 2

        ) * scale_y


        # Clamp
        x1 = max(
            0,
            min(
                x1,
                original_width - 1
            )
        )


        y1 = max(
            0,
            min(
                y1,
                original_height - 1
            )
        )


        x2 = max(
            0,
            min(
                x2,
                original_width - 1
            )
        )


        y2 = max(
            0,
            min(
                y2,
                original_height - 1
            )
        )


        results.append(

            {

                "bbox": [

                    x1,
                    y1,
                    x2,
                    y2

                ],

                "confidence":
                    confidence,

                "class":
                    class_name

            }

        )


    return results


# ============================================================
# 21) DRAW TRACKS
# ============================================================

def draw_tracks(

    frame,

    tracker,

    prefix,

    color

):

    count = 0


    for track in tracker.tracks:


        if track.lost > MAX_LOST_FRAMES:
            continue


        x1, y1, x2, y2 = (

            track.bbox
            .astype(int)

        )


        # Keep box inside frame
        x1 = max(
            0,
            min(
                x1,
                frame_width - 1
            )
        )

        y1 = max(
            0,
            min(
                y1,
                frame_height - 1
            )
        )

        x2 = max(
            0,
            min(
                x2,
                frame_width - 1
            )
        )

        y2 = max(
            0,
            min(
                y2,
                frame_height - 1
            )
        )


        # ----------------------------------------------------
        # BOX
        # ----------------------------------------------------

        cv2.rectangle(

            frame,

            (
                x1,
                y1
            ),

            (
                x2,
                y2
            ),

            color,

            2

        )


        # ----------------------------------------------------
        # LABEL
        # ----------------------------------------------------

        label = (

            f"{prefix} "

            f"ID:{track.track_id} "

            f"{track.class_name} "

            f"{track.confidence:.0%}"

        )


        cv2.putText(

            frame,

            label,

            (
                x1,
                max(
                    20,
                    y1 - 8
                )
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.48,

            color,

            2,

            cv2.LINE_AA

        )


        count += 1


    return count


# ============================================================
# 22) STATISTICS
# ============================================================

statistics = {

    "uniform":
        Counter(),

    "weapon":
        Counter(),

    "vehicle":
        Counter()

}


confidence_statistics = {

    "uniform":
        defaultdict(list),

    "weapon":
        defaultdict(list),

    "vehicle":
        defaultdict(list)

}


# ============================================================
# 23) MAIN LOOP
# ============================================================

print("\n" + "=" * 70)
print("STARTING AI DETECTION + TRACKING")
print("=" * 70)


frame_number = 0

detection_frames = 0

start_time = time.time()


while True:


    ret, frame = cap.read()


    if not ret:
        break


    frame_number += 1


    # ========================================================
    # PREDICT TRACKS
    # ========================================================

    uniform_tracker.predict()

    weapon_tracker.predict()

    vehicle_tracker.predict()


    # ========================================================
    # SHOULD RUN AI?
    # ========================================================

    run_detection = (

        (frame_number - 1)
        % DETECTION_INTERVAL
        == 0

    )


    if run_detection:


        detection_frames += 1


        # ----------------------------------------------------
        # RESIZE
        # ----------------------------------------------------

        api_frame = (
            resize_for_api(
                frame
            )
        )


        api_height, api_width = (
            api_frame.shape[:2]
        )


        # ----------------------------------------------------
        # ENCODE
        # ----------------------------------------------------

        image_bytes = (
            encode_frame(
                api_frame
            )
        )


        # ====================================================
        # THREE MODELS
        # ====================================================

        model_configs = {

            "uniform":
                UNIFORM_MODEL,

            "weapon":
                WEAPON_MODEL,

            "vehicle":
                VEHICLE_MODEL

        }


        results = {}


        # ====================================================
        # PARALLEL API CALLS
        # ====================================================

        with ThreadPoolExecutor(

            max_workers=MAX_WORKERS

        ) as executor:


            futures = {

                executor.submit(

                    run_roboflow,

                    image_bytes,

                    model_id

                ): name

                for name, model_id
                in model_configs.items()

            }


            for future in as_completed(
                futures
            ):


                name = futures[
                    future
                ]


                try:

                    results[
                        name
                    ] = future.result()


                except Exception as e:

                    results[
                        name
                    ] = {

                        "predictions": [],

                        "error":
                            str(e)

                    }


        # ====================================================
        # CONVERT RESULTS
        # ====================================================

        uniform_detections = (
            convert_predictions(

                results.get(
                    "uniform",
                    {}
                ).get(
                    "predictions",
                    []
                ),

                frame_width,

                frame_height,

                api_width,

                api_height

            )
        )


        weapon_detections = (
            convert_predictions(

                results.get(
                    "weapon",
                    {}
                ).get(
                    "predictions",
                    []
                ),

                frame_width,

                frame_height,

                api_width,

                api_height

            )
        )


        vehicle_detections = (
            convert_predictions(

                results.get(
                    "vehicle",
                    {}
                ).get(
                    "predictions",
                    []
                ),

                frame_width,

                frame_height,

                api_width,

                api_height

            )
        )


        # ====================================================
        # UPDATE TRACKERS
        # ====================================================

        uniform_tracker.update(
            uniform_detections
        )


        weapon_tracker.update(
            weapon_detections
        )


        vehicle_tracker.update(
            vehicle_detections
        )


        # ====================================================
        # STATISTICS
        # ====================================================

        for d in uniform_detections:

            cls = d["class"]

            statistics[
                "uniform"
            ][cls] += 1

            confidence_statistics[
                "uniform"
            ][cls].append(
                d["confidence"]
            )


        for d in weapon_detections:

            cls = d["class"]

            statistics[
                "weapon"
            ][cls] += 1

            confidence_statistics[
                "weapon"
            ][cls].append(
                d["confidence"]
            )


        for d in vehicle_detections:

            cls = d["class"]

            statistics[
                "vehicle"
            ][cls] += 1

            confidence_statistics[
                "vehicle"
            ][cls].append(
                d["confidence"]
            )


    # ========================================================
    # DRAW
    # ========================================================

    uniform_count = draw_tracks(

        frame,

        uniform_tracker,

        "UNIFORM",

        (255, 0, 0)

    )


    weapon_count = draw_tracks(

        frame,

        weapon_tracker,

        "WEAPON",

        (0, 0, 255)

    )


    vehicle_count = draw_tracks(

        frame,

        vehicle_tracker,

        "VEHICLE",

        (0, 255, 0)

    )


    # ========================================================
    # HEADER
    # ========================================================

    cv2.rectangle(

        frame,

        (0, 0),

        (520, 105),

        (0, 0, 0),

        -1

    )


    cv2.putText(

        frame,

        "AI SECURITY SURVEILLANCE",

        (10, 25),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.60,

        (255, 255, 255),

        2,

        cv2.LINE_AA

    )


    cv2.putText(

        frame,

        f"Frame: {frame_number}",

        (10, 50),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.50,

        (255, 255, 255),

        1,

        cv2.LINE_AA

    )


    cv2.putText(

        frame,

        f"Uniform: {uniform_count}",

        (10, 78),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.50,

        (255, 255, 255),

        1,

        cv2.LINE_AA

    )


    cv2.putText(

        frame,

        f"Weapon: {weapon_count}",

        (180, 78),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.50,

        (255, 255, 255),

        1,

        cv2.LINE_AA

    )


    cv2.putText(

        frame,

        f"Vehicle: {vehicle_count}",

        (350, 78),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.50,

        (255, 255, 255),

        1,

        cv2.LINE_AA

    )


    # ========================================================
    # WRITE
    # ========================================================

    writer.write(
        frame
    )


    # ========================================================
    # PROGRESS
    # ========================================================

    if frame_number % 20 == 0:


        elapsed = (

            time.time()
            - start_time

        )


        processing_fps = (

            frame_number
            /
            elapsed

            if elapsed > 0

            else 0

        )


        progress = (

            frame_number
            /
            total_frames
            *
            100

            if total_frames > 0

            else 0

        )


        print(

            f"\rProgress: "
            f"{progress:6.2f}% | "

            f"Frame: "
            f"{frame_number:5d} | "

            f"Speed: "
            f"{processing_fps:5.2f} FPS | "

            f"Tracks: "
            f"{uniform_count + weapon_count + vehicle_count:3d}",

            end=""

        )


# ============================================================
# 24) RELEASE
# ============================================================

cap.release()

writer.release()


processing_time = (

    time.time()
    - start_time

)


print("\n")


# ============================================================
# 25) FINAL VIDEO ENCODING
# ============================================================

print("=" * 70)
print("ENCODING FINAL VIDEO")
print("=" * 70)


ffmpeg_command = (

    f'ffmpeg -y '

    f'-i "{RAW_OUTPUT_VIDEO}" '

    f'-c:v libx264 '

    f'-preset veryfast '

    f'-crf 23 '

    f'-pix_fmt yuv420p '

    f'-movflags +faststart '

    f'"{FINAL_OUTPUT_VIDEO}" '

    f'-loglevel error'

)


ffmpeg_status = os.system(
    ffmpeg_command
)


if (

    ffmpeg_status != 0

    or

    not FINAL_OUTPUT_VIDEO.exists()

):

    print(
        "⚠️ FFmpeg encoding failed."
    )

    print(
        "Using raw output instead."
    )

    FINAL_OUTPUT_VIDEO = (
        RAW_OUTPUT_VIDEO
    )


# ============================================================
# 26) FINAL STATISTICS
# ============================================================

print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)


print(
    "\nTotal frames:",
    frame_number
)

print(
    "AI detection frames:",
    detection_frames
)

print(
    "Processing time:",
    f"{processing_time:.2f} sec"
)

print(
    "Average processing FPS:",
    f"{frame_number / processing_time:.2f}"
)


# ============================================================
# 27) UNIFORM RESULTS
# ============================================================

print("\n" + "-" * 60)
print("UNIFORM RESULTS")
print("-" * 60)


if statistics["uniform"]:

    for cls, count in (
        statistics[
            "uniform"
        ].most_common()
    ):

        values = (
            confidence_statistics[
                "uniform"
            ][cls]
        )

        avg_conf = (

            sum(values)
            /
            len(values)

        )

        print(

            f"{cls:25s} "
            f"Count={count:5d} "
            f"AvgConf={avg_conf:.2%}"

        )

else:

    print(
        "No uniform detections."
    )


# ============================================================
# 28) WEAPON RESULTS
# ============================================================

print("\n" + "-" * 60)
print("WEAPON RESULTS")
print("-" * 60)


if statistics["weapon"]:

    for cls, count in (
        statistics[
            "weapon"
        ].most_common()
    ):

        values = (
            confidence_statistics[
                "weapon"
            ][cls]
        )

        avg_conf = (

            sum(values)
            /
            len(values)

        )

        print(

            f"{cls:25s} "
            f"Count={count:5d} "
            f"AvgConf={avg_conf:.2%}"

        )

else:

    print(
        "No weapon detections."
    )


# ============================================================
# 29) VEHICLE RESULTS
# ============================================================

print("\n" + "-" * 60)
print("VEHICLE RESULTS")
print("-" * 60)


if statistics["vehicle"]:

    for cls, count in (
        statistics[
            "vehicle"
        ].most_common()
    ):

        values = (
            confidence_statistics[
                "vehicle"
            ][cls]
        )

        avg_conf = (

            sum(values)
            /
            len(values)

        )

        print(

            f"{cls:25s} "
            f"Count={count:5d} "
            f"AvgConf={avg_conf:.2%}"

        )

else:

    print(
        "No vehicle detections."
    )


# ============================================================
# 30) DISPLAY FINAL VIDEO
# ============================================================

print("\n" + "=" * 70)
print("FINAL VIDEO")
print("=" * 70)


video_bytes = (
    FINAL_OUTPUT_VIDEO.read_bytes()
)


video_base64 = (
    base64.b64encode(
        video_bytes
    )
    .decode()
)


display(

    HTML(

        f"""
        <video
            width="900"
            controls
            style="max-width:100%;"
        >

            <source
                src="data:video/mp4;base64,{video_base64}"
                type="video/mp4"
            >

        </video>
        """

    )

)


# ============================================================
# 31) DOWNLOAD
# ============================================================

files.download(
    str(FINAL_OUTPUT_VIDEO)
)


print("\n" + "=" * 70)
print("✅ SYSTEM COMPLETED")
print("=" * 70)
print()
print("Uniform  : Detection + Tracking")
print("Weapon   : Detection + Tracking")
print("Vehicle  : Detection + Tracking")
print()
print("Detection interval:",
      DETECTION_INTERVAL)

print("Tracking:",
      "Every frame")

print("=" * 70)